# **Реализация 3д модели центральной линии сосуда сердца**

In [1]:
import numpy as np
from Reconstruction_coronary_arteria.Reconstruction import Reconstruction_methods
from Reconstruction_coronary_arteria.open_json import GeneratedDataset
from torchvision import transforms as T
import cv2
import torch
from matplotlib import pyplot as plt
from skimage.morphology import skeletonize
from Reconstruction_coronary_arteria.Transformation import TransformationMatrix
from scipy.spatial import distance
import uuid
import plotly.graph_objects as go 
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'Reconstruction_coronary_arteria'

In [ ]:
file_path1 = '/home/alexus/Desktop/For_Generator/centerline1.npy'
file_path2 = '/home/alexus/Desktop/For_Generator/centerline2.npy'
# Чтение файла
data1 = np.load(file_path1, allow_pickle=True)
data2 = np.load(file_path2, allow_pickle=True)

# Вывод x,y центральной линии 1 снимка
centerline1 = data1.item()['Paths'] # centerline1[vessel index][point][x,y]
centerline2 = data2.item()['Paths']
centerlines = [centerline1,centerline2]

theta1 = data1.item()['PositionerPrimaryAngle'] 
theta2 = data2.item()['PositionerPrimaryAngle']
phi1 = data1.item()['PositionerSecondaryAngle'] # все углы в градусах
phi2 = data2.item()['PositionerSecondaryAngle']
r1 = data1.item()['DistanceSourceToPatient'] # в мм
r2 = data2.item()['DistanceSourceToPatient'] # в мм
radius1 = data1.item()['Radius']
radius2 = data2.item()['Radius'] # радиусы сосудов к 3 веткам
print("Углы поворота ангиографа в левосторонней ДСК:") 
print(f'Первая проекция: theta = {theta1}, phi = {phi1}, \nВторая проекция: theta = {theta2}, phi = {phi2}') 
print(f'\nРасстояние от источника до пациента:')
print(f'Первая проекция: focal_point = {r1} \nВторая проекция: focal_point = {r2}')
# Фокальные точки (источники рентгеновских лучей)
focal_points = [
    np.array([r1 * np.sin(np.radians(phi1)) * np.cos(np.radians(theta1)), 
              r1 * np.sin(np.radians(phi1)) * np.sin(np.radians(theta1)),
              r1 * np.cos(np.radians(phi1))]),  # Проекция 1 (theta - азимутальный угол, вращается вокруг Z, phi - полярный угол, вращается вокруг X)
    np.array([r2 * np.sin(np.radians(phi2)) * np.cos(np.radians(theta2)), 
              r2 * np.sin(np.radians(phi2)) * np.sin(np.radians(theta2)),
              r2 * np.cos(np.radians(phi2))]),  # Проекция 2 
]

colors = ['blue', 'red', 'green']  # Цвета для разных веток
index = 1
for vessel_idx in range(len(centerlines[index])):  # Проходим по всем веткам
    # Извлекаем координаты x и y для текущей ветки
    x = [point[0] for point in centerlines[index][vessel_idx]]
    y = [point[1] for point in centerlines[index][vessel_idx]]
    
    # Рисуем линию с точками
    plt.plot(x, y, color=colors[vessel_idx], label=f'Vessel {vessel_idx}', marker='o', markersize=4)


In [ ]:
len(centerline1[0])

In [ ]:
len(data1.item()['Radius'][0])

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import interp1d

# Чтение данных
file_path1 = '/home/alexus/Desktop/For_Generator/centerline1.npy'
file_path2 = '/home/alexus/Desktop/For_Generator/centerline2.npy'
data1 = np.load(file_path1, allow_pickle=True)
data2 = np.load(file_path2, allow_pickle=True)

centerline1 = data1.item()['Paths']
centerline2 = data2.item()['Paths']
centerlines = [centerline1, centerline2]

theta1 = data1.item()['PositionerPrimaryAngle']  # -1.0°
theta2 = data2.item()['PositionerPrimaryAngle']  # 46.5°
phi1 = data1.item()['PositionerSecondaryAngle']  # 0.7°
phi2 = data2.item()['PositionerSecondaryAngle']  # -0.7°
r1 = data1.item()['DistanceSourceToPatient']  # 859.0167313 мм
r2 = data2.item()['DistanceSourceToPatient']  # 893.3682903 мм

# Фокальные точки
focal_points = [
    np.array([
        r1 * np.sin(np.radians(phi1)) * np.cos(np.radians(theta1)),
        r1 * np.sin(np.radians(phi1)) * np.sin(np.radians(theta1)),
        r1 * np.cos(np.radians(phi1))
    ]),
    np.array([
        r2 * np.sin(np.radians(phi2)) * np.cos(np.radians(theta2)),
        r2 * np.sin(np.radians(phi2)) * np.sin(np.radians(theta2)),
        r2 * np.cos(np.radians(phi2))
    ])
]

# Ресемплирование центральных линий
def resample_centerline(centerline, num_points):
    points = np.array(centerline)
    distances = np.cumsum(np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1)))
    distances = np.insert(distances, 0, 0)
    t = np.linspace(0, distances[-1], num_points)
    fx = interp1d(distances, points[:, 0], kind='linear')
    fy = interp1d(distances, points[:, 1], kind='linear')
    new_points = np.vstack((fx(t), fy(t))).T
    return new_points.tolist()

num_points = 100
centerlines_resampled = [[], []]
for proj_idx in range(2):
    for vessel_idx in range(len(centerlines[proj_idx])):
        resampled = resample_centerline(centerlines[proj_idx][vessel_idx], num_points)
        centerlines_resampled[proj_idx].append(resampled)
centerlines = centerlines_resampled

# Параметры ангиографа
SID = 1107.0  # Расстояние от источника до детектора (мм)
pixel_size = 0.2  # мм/пиксель
image_size = [2048, 2048]  # Пробуем 2048x2048, так как координаты выходят за 1024
principal_point = [image_size[0] / 2, image_size[1] / 2]  # [1024, 1024]

# Изоцентр
iso_center = np.array([50, 0, 859])  # Смещён на Z = 859 мм

# Матрицы проекции
def get_projection_matrix(focal_point, theta, phi):
    # Направление от фокальной точки к изоцентру
    direction = (iso_center - focal_point) / np.linalg.norm(iso_center - focal_point)
    
    # Позиция плоскости проекции (детектора)
    detector_distance = SID
    detector_center = focal_point + detector_distance * direction
    
    # Ориентация плоскости (перпендикулярно направлению)
    z_axis = direction  # Направление от источника к детектору
    x_axis = np.array([1, 0, 0])  # Произвольная ось X
    if np.abs(np.dot(z_axis, x_axis)) > 0.99:  # Если z_axis почти параллелен x_axis
        x_axis = np.array([0, 1, 0])
    x_axis = x_axis - np.dot(x_axis, z_axis) * z_axis
    x_axis = x_axis / np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    y_axis = y_axis / np.linalg.norm(y_axis)
    
    # Матрица вращения (ориентация детектора)
    R = np.vstack((x_axis, y_axis, z_axis)).T
    
    # Вектор трансляции
    t = -focal_point
    
    # Внутренняя калибровочная матрица K
    focal_length = SID / pixel_size  # Фокусное расстояние в пикселях: 1107 / 0.2 = 5535
    K = np.array([
        [focal_length, 0, principal_point[0]],
        [0, focal_length, principal_point[1]],
        [0, 0, 1]
    ])
    
    # Матрица проекции P = K * [R | t]
    Rt = np.hstack((R, t.reshape(3, 1)))
    P = K @ Rt
    return P

P1 = get_projection_matrix(focal_points[0], theta1, phi1)
P2 = get_projection_matrix(focal_points[1], theta2, phi2)

# Преобразование координат центральных линий из мм в пиксели
centerlines_pixel = [[], []]
for proj_idx in range(2):
    for vessel_idx in range(len(centerlines[proj_idx])):
        points = np.array(centerlines[proj_idx][vessel_idx])
        points_pixel = points / pixel_size  # Переводим мм в пиксели
        # Центрируем координаты относительно principal_point
        points_pixel[:, 0] -= principal_point[0]
        points_pixel[:, 1] -= principal_point[1]
        centerlines_pixel[proj_idx].append(points_pixel.tolist())
centerlines = centerlines_pixel

# Отладочная информация: диапазон координат центральных линий после центрирования
for proj_idx in range(2):
    for vessel_idx in range(len(centerlines[proj_idx])):
        points = np.array(centerlines[proj_idx][vessel_idx])
        print(f"Projection {proj_idx}, Vessel {vessel_idx}: X range = {points[:, 0].min()} to {points[:, 0].max()}, Y range = {points[:, 1].min()} to {points[:, 1].max()} (pixels, centered)")

# Триангуляция
def triangulate_point(p1, p2, P1, P2):
    x1, y1 = p1
    x2, y2 = p2
    
    A = np.zeros((4, 4))
    A[0] = x1 * P1[2] - P1[0]
    A[1] = y1 * P1[2] - P1[1]
    A[2] = x2 * P2[2] - P2[0]
    A[3] = y2 * P2[2] - P2[1]
    
    _, _, V = np.linalg.svd(A)
    X = V[-1]
    X = X / X[3]
    return X[:3]

# Реконструкция 3D центральной линии
centerline_3d = []
for vessel_idx in range(len(centerlines[0])):
    vessel_3d = []
    for i in range(len(centerlines[0][vessel_idx])):
        p1 = centerlines[0][vessel_idx][i]
        p2 = centerlines[1][vessel_idx][i]
        point_3d = triangulate_point(p1, p2, P1, P2)
        vessel_3d.append(point_3d)
    centerline_3d.append(np.array(vessel_3d))
    print(f"Vessel {vessel_idx}: Reconstructed {len(vessel_3d)} points")
    # Отладочная информация: диапазон 3D координат
    vessel_3d = np.array(vessel_3d)
    print(f"Vessel {vessel_idx}: X range = {vessel_3d[:, 0].min()} to {vessel_3d[:, 0].max()}, Y range = {vessel_3d[:, 1].min()} to {vessel_3d[:, 1].max()}, Z range = {vessel_3d[:, 2].min()} to {vessel_3d[:, 2].max()} (mm)")


fig = go.Figure()
colors = ['blue', 'red', 'green']
for i, vessel in enumerate(centerline_3d):
    if len(vessel) > 0:
        fig.add_trace(go.Scatter3d(
            x=vessel[:, 0], y=vessel[:, 1], z=vessel[:, 2],
            mode='lines+markers', name=f'Vessel {i}',
            line=dict(color=colors[i], width=4),
            marker=dict(size=3)
        ))

fig.update_layout(
    title="3D Centerline Reconstruction",
    scene=dict(
        xaxis_title="X (mm) (Right to Left)",
        yaxis_title="Y (mm) (Back to Front)",
        zaxis_title="Z (mm) (Feet to Head)",
        aspectmode='cube'
    ),
    width=800, height=800
)
fig.show()

In [ ]:
centerline_3d